In [4]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import itertools


# ══════════════════════════════════════════════════════════════════════════════
# METRICS  (inlined — no sentinel package needed)
# Source: ESA Anomaly Detection Benchmark
# ══════════════════════════════════════════════════════════════════════════════

def find_anomaly_segments(y: np.ndarray) -> list:
    """Return list of {start, end} dicts for contiguous runs of 1s in y."""
    padded = np.concatenate(([0], y.astype(np.int8), [0]))
    d      = np.diff(padded)
    starts = np.where(d ==  1)[0]
    ends   = np.where(d == -1)[0] - 1
    return [{"start": int(s), "end": int(e)} for s, e in zip(starts, ends)]


def _find_predicted_segments(y_pred: np.ndarray) -> list:
    """Return contiguous predicted-anomaly segments."""
    padded = np.concatenate(([0], y_pred.astype(np.int8), [0]))
    d      = np.diff(padded)
    starts = np.where(d ==  1)[0]
    ends   = np.where(d == -1)[0] - 1
    return [{"start": int(s), "end": int(e)} for s, e in zip(starts, ends)]


def corrected_event_f05(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    beta: float = 0.5,
) -> dict:
    """
    Corrected event-wise F-beta score (ESA ADB metric).

    Re_e   = TP_events / (TP_events + FN_events)
    Pr_ew  = TP_events / (TP_events + FP_pred_events)
    TNR    = 1 - fp_samples / N_nominal
    Pr_c   = Pr_ew x TNR
    F_beta = (1+beta^2) * Pr_c * Re_e / (beta^2 * Pr_c + Re_e)
    """
    y_true = np.asarray(y_true, dtype=np.int8)
    y_pred = np.asarray(y_pred, dtype=np.int8)

    true_segs  = find_anomaly_segments(y_true)
    pred_segs  = _find_predicted_segments(y_pred)

    n_nominal  = int((y_true == 0).sum())
    n_events   = len(true_segs)

    if n_events == 0:
        return {"f_score": 0.0, "precision": 0.0, "recall": 0.0,
                "tp_events": 0, "fn_events": 0,
                "fp_pred_events": len(pred_segs), "fp_samples": 0, "tnr": 1.0}

    # TP / FN over ground-truth segments
    tp_events  = 0
    fn_events  = 0
    matched_pred = [False] * len(pred_segs)

    for ts in true_segs:
        detected = False
        for p, ps in enumerate(pred_segs):
            if ps["end"] >= ts["start"] and ps["start"] <= ts["end"]:
                matched_pred[p] = True
                detected = True
        if detected:
            tp_events += 1
        else:
            fn_events += 1

    # FP predicted events (no GT overlap)
    fp_pred_events = sum(1 for m in matched_pred if not m)

    # TNR correction
    fp_samples = int(((y_pred == 1) & (y_true == 0)).sum())
    tnr        = (1.0 - fp_samples / n_nominal) if n_nominal > 0 else 1.0

    # Corrected precision
    denom_pr  = tp_events + fp_pred_events
    pr_ew     = (tp_events / denom_pr) if denom_pr > 0 else 0.0
    precision = pr_ew * tnr

    # Recall & F-beta
    recall  = tp_events / n_events
    if precision + recall == 0:
        f_score = 0.0
    else:
        f_score = (1 + beta**2) * precision * recall / (beta**2 * precision + recall)

    return {
        "f_score"       : round(f_score,   6),
        "precision"     : round(precision, 6),
        "recall"        : round(recall,    6),
        "tp_events"     : tp_events,
        "fn_events"     : fn_events,
        "fp_pred_events": fp_pred_events,
        "fp_samples"    : fp_samples,
        "tnr"           : round(tnr,       6),
    }


# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════

VAL_FRAC    = 0.2
Q_LOW_GRID  = [0.0001, 0.0002, 0.0003, 0.0004, 0.0005]
Q_HIGH_GRID = [0.99, 0.995, 0.996, 0.997, 0.998]

DATA_DIR   = "/Users/alex/Downloads/esa-adb-challenge (1)"
train_path = f"{DATA_DIR}/train.parquet"
test_path  = f"{DATA_DIR}/test.parquet"

column_names = ['channel_' + str(n) for n in range(41, 47)]


# ══════════════════════════════════════════════════════════════════════════════
# 1.  LOAD
# ══════════════════════════════════════════════════════════════════════════════

cols  = ['id', 'is_anomaly'] + column_names

train = pq.read_table(train_path, columns=cols).to_pandas().set_index('id')
test  = pq.read_table(test_path,  columns=['id'] + column_names).to_pandas().set_index('id')

print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")


# ══════════════════════════════════════════════════════════════════════════════
# 2.  TEMPORAL SPLIT  (no shuffle)
# ══════════════════════════════════════════════════════════════════════════════

split_idx = int(len(train) * (1 - VAL_FRAC))

train_tr  = train.iloc[:split_idx]
train_val = train.iloc[split_idx:]

X_tr  = train_tr[column_names]
X_val = train_val[column_names]
y_val = train_val['is_anomaly'].values

print(f"\nTrain split : {len(train_tr):,} rows")
print(f"Val split   : {len(train_val):,} rows  "
      f"(anomaly rate: {y_val.mean()*100:.1f}%)")


# ══════════════════════════════════════════════════════════════════════════════
# 3.  THRESHOLD FUNCTION
# ══════════════════════════════════════════════════════════════════════════════

def predict_with_thresholds(df, low, high):
    return ((df < low) | (df > high)).any(axis=1).astype(int).values


# ══════════════════════════════════════════════════════════════════════════════
# 4.  GRID SEARCH ON VALIDATION SET
# ══════════════════════════════════════════════════════════════════════════════

best_score      = -1
best_params     = None
best_thresholds = None
best_metrics    = None

grid = list(itertools.product(Q_LOW_GRID, Q_HIGH_GRID))

print(f"Running grid search ({len(grid)} combinations) ...")
for q_low, q_high in grid:

    low  = X_tr.quantile(q_low)
    high = X_tr.quantile(q_high)

    y_pred_val = predict_with_thresholds(X_val, low, high)

    metric        = corrected_event_f05(y_val, y_pred_val)
    metric["f1"]  = corrected_event_f05(y_val, y_pred_val, beta=1.0)["f_score"]
    score         = metric["f_score"]

    if score > best_score:
        best_score      = score
        best_params     = (q_low, q_high)
        best_thresholds = (low, high)
        best_metrics    = metric

print(f"\nBest params : q_low={best_params[0]}, q_high={best_params[1]}")
print(f"Best F0.5   : {best_score:.4f}")
print(f"Metrics     : {best_metrics}")


# ══════════════════════════════════════════════════════════════════════════════
# 5.  REFIT THRESHOLDS ON FULL TRAIN
# ══════════════════════════════════════════════════════════════════════════════
# Now that best params are found on val, refit on ALL training data
# so the final model has seen as much normal data as possible.

low  = train[column_names].quantile(best_params[0])
high = train[column_names].quantile(best_params[1])

print(f"\nFinal thresholds (fit on full train):")
print(f"  Low  (q={best_params[0]}) : {low.to_dict()}")
print(f"  High (q={best_params[1]}) : {high.to_dict()}")


# ══════════════════════════════════════════════════════════════════════════════
# 6.  INFERENCE ON TEST
# ══════════════════════════════════════════════════════════════════════════════

y_pred_test = predict_with_thresholds(test[column_names], low, high)


# ══════════════════════════════════════════════════════════════════════════════
# 7.  BUILD SUBMISSION
# ══════════════════════════════════════════════════════════════════════════════

# Auto-detect expected column name from sample submission if available
try:
    sample   = pd.read_parquet(f"{DATA_DIR}/sample_submission.parquet")
    pred_col = [c for c in sample.columns if c != 'id'][0]
    print(f"\nSubmission column detected from sample: '{pred_col}'")
except Exception:
    pred_col = "anomaly_flag"
    print(f"\nNo sample submission found — using column: '{pred_col}'")

submission = pd.DataFrame({
    'id':     test.index,
    pred_col: y_pred_test,
})


# ══════════════════════════════════════════════════════════════════════════════
# 8.  SANITY CHECKS
# ══════════════════════════════════════════════════════════════════════════════

print(f"\nSubmission shape    : {submission.shape}")
print(f"Anomalous timesteps : {submission[pred_col].sum():,}  "
      f"({submission[pred_col].mean()*100:.2f}%)")
print(f"\nFirst 5 rows:")
print(submission.head())


# ══════════════════════════════════════════════════════════════════════════════
# 9.  SAVE
# ══════════════════════════════════════════════════════════════════════════════

out_path = "submission_quantile_threshold.parquet"
submission.to_parquet(out_path, index=False)

print(f"\n✓ Saved → {out_path}")
print(f"  Upload this file to Kaggle.")
print(f"\nValidation metrics summary:")
print(f"  F0.5      : {best_score:.4f}")
print(f"  Precision : {best_metrics['precision']:.4f}")
print(f"  Recall    : {best_metrics['recall']:.4f}")
print(f"  F1        : {best_metrics['f1']:.4f}")
print(f"  TP events : {best_metrics['tp_events']}")
print(f"  FN events : {best_metrics['fn_events']}")
print(f"  FP events : {best_metrics['fp_pred_events']}")

Train : (14728321, 7)
Test  : (521280, 6)

Train split : 11,782,656 rows
Val split   : 2,945,665 rows  (anomaly rate: 10.5%)
Running grid search (25 combinations) ...

Best params : q_low=0.0001, q_high=0.997
Best F0.5   : 0.6977
Metrics     : {'f_score': 0.697674, 'precision': 1.0, 'recall': 0.315789, 'tp_events': 12, 'fn_events': 26, 'fp_pred_events': 0, 'fp_samples': 0, 'tnr': 1.0, 'f1': 0.48}

Final thresholds (fit on full train):
  Low  (q=0.0001) : {'channel_41': 0.0, 'channel_42': 0.0, 'channel_43': 0.0, 'channel_44': 0.0, 'channel_45': 0.0, 'channel_46': 0.0}
  High (q=0.997) : {'channel_41': 0.8282751441001892, 'channel_42': 0.8032125234603882, 'channel_43': 0.7885160446166992, 'channel_44': 0.8133468627929688, 'channel_45': 0.8321169018745422, 'channel_46': 0.7872520685195923}

Submission column detected from sample: 'is_anomaly'

Submission shape    : (521280, 2)
Anomalous timesteps : 19  (0.00%)

First 5 rows:
         id  is_anomaly
0  14728321           0
1  14728322     